# Setup

## Environment

In [ ]:
# %%bash
# 1. Parameterize target branch name
BRANCH="LLM"
MODEL = "Gemini"

# 2. Reset workspace directory for a clean slate
!rm -rf FoSpy

# 3. Fast clone single branch
!git clone --single-branch --branch $BRANCH https://github.com/errthumt/FoSpy.git FoSpy

# 4. Change notebook working directory to target folder
%cd FoSpy/LLM/sandbox

# 5. Fast dependency installation using uv without touching native PyTorch
%pip install -q uv
!uv pip install -r colab_reqs.txt #--no-deps
#!uv pip install -q safetensors fsspec huggingface_hub


## Python Imports

In [ ]:
import json
import os
import re
from pathlib import Path
import google.generativeai as genai
from google.colab import userdata

## Globals and Helpers

In [ ]:
INPUT_SOURCES = {
    "Ba2Zn5As6 Supplemental Information": (
        "Ba2Zn5As6_SI.txt",
        "Solid-State Material Science",
    )
}

# Define default path pointers based on your setup variables
MODEL = "Gemini"
INPUT_DIR = Path("inputs")
OUTPUT_FILE = Path(MODEL) / "output" / "tokens.json"

# Configure the API connection safely using your Colab Secrets vault
try:
    GOOGLE_API_KEY = userdata.get('FoSpy_Testing_API_key')
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception:
    print("ERROR: Please add your FoSpy_Testing_API_key to the Colab Secrets sidebar panel.")


def get_prompt(input_text, context="None Specified"):
    return f"""
    Extract the fundamental scientific information pertaining to the synthetic
    method and product characterization described in the following input. Return
    the result as a JSON-formatted list of atomic, order-independent scientific information
    tokens. Each token must represent one fact, relationship or entity. Whenever
    possible, token values should be coercible to primitive data types,
    including separating quantities into a `value` and `units` fields. If
    specified, context refers to the scientific field or chemical category
    pertaining to the intended synthetic products.

    Rules:
    - Remove redundancy.
    - Normalize chemical formulas.
    - Normalize references to individual elements (outside chemical formulas) to their full element name.
    - Each token must be independent of sentence order.
    - Token meaning must be independent of narrative context.
    - A preamble before JSON output is optional, but must not contain any JSON-signaling characters such as {{ or [.
    - No further response characters are allowed after the JSON output.

    Input:
    \"\"\"
    {input_text}
    \"\"\"

    Context:
    \"\"\"
    {context}
    \"\"\"
    """


# Get Model

## Build Model

In [ ]:
print("Initializing Gemini Cloud Client (gemini-2.5-flash)...")
model = genai.GenerativeModel('gemini-2.5-flash')
print("Model configuration loaded and ready for execution!")


# Main Logic

In [ ]:
# Ensure the workspace target directory physically exists
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

out = {"tokens": {}, "full_responses": {}}
out_tokens = out["tokens"]
out_full = out["full_responses"]

for name, (file, context) in INPUT_SOURCES.items():
    target_file = INPUT_DIR / file
    if not target_file.exists():
        print(f"ERROR: Target input file missing at path: {target_file}")
        continue

    input_text = target_file.read_text()
    prompt = get_prompt(input_text, context)

    print(f"Sending text generation request for: {name}")
    # Force Gemini to strictly output structured schema data natively
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"}
    )
    
    raw = response.text
    out_full[name] = raw

    # Validate the JSON output structure
    matched = re.search(r"[\{\[].*[\}\]]", raw, flags=re.DOTALL)
    if not matched:
        print(f"ERROR: No JSON array or object pattern isolated for {name}")
        continue

    json_str = matched.group(0)
    try:
        out_tokens[name] = json.loads(json_str)
    except json.JSONDecodeError:
        print(f"ERROR: Truncated or invalid JSON block isolated: {json_str}")

with OUTPUT_FILE.open("w") as f:
    json.dump(out, f, indent=4)

print(f"Pipeline executed successfully. Extraction matrix written to: {OUTPUT_FILE}")
